In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import os
from pathlib import Path
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from shapely.geometry import box, mapping
import pyproj
import geopandas as gpd
from scipy import stats

import sys
sys.path.append('../../')
from renewable_data_load import *
from plotting_config import model_colors, gwl_colors, model_markers

plt.style.use('../../renewable_analysis.mplstyle')

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

# High Demand - Low Supply Coincidence Analysis

This notebook analyzes the overlap between **high electricity demand days** and **renewable resource drought days** (low generation), examining how often these critical events coincide and whether their relationship changes in future warming scenarios.

## Research Questions

1. **How often do high demand days overlap with low generation days?**
   - For each resource: PV, onshore wind, offshore wind
   - By season: JFM, AMJ, JAS, OND
   - Across utility regions in California

2. **How does this overlap change in the future?**
   - Compare GWL 0.8°C (reference) to GWL 2.0°C (future)
   - Quantify absolute and relative changes

3. **Is the coincidence statistically significant?**
   - Test if observed coincidence exceeds what would be expected by chance (independence)
   - Determine if high demand and low supply are becoming more coupled

4. **Which resource type has the strongest relationship with high demand?**
   - Compare PV vs onshore wind vs offshore wind
   - Examine seasonal differences in these relationships

## Approach

**Data Sources:**
- High demand binary masks (SEI > 1.64 threshold)
- Resource drought binary masks (below 10th percentile capacity factor)
- 4 climate models × 2 GWLs × 9 California regions

**Methodology:**
- Calculate observed coincidence days per season
- Calculate expected coincidence under independence assumption
- Test statistical significance of deviation from independence
- Decompose future changes into frequency vs coupling components

**Seasons:**
- JFM: January, February, March (Winter)
- AMJ: April, May, June (Spring)  
- JAS: July, August, September (Summer)
- OND: October, November, December (Fall)

## Setup and Configuration

In [ ]:
# Analysis parameters
target_gwls = [0.8, 2.0]
simulations = ["mpi-esm1-2-hr", "miroc6", "taiesm1", "ec-earth3"]
seasons = ['JFM', 'AMJ', 'JAS', 'OND']

# Utility regions to analyze (using standardized demand mask names)
utility_regions = ["PGE", "SCE", "SDGE", "IID", "LDWP", "NCNC", "WECC-NW", "WECC-SW", 'WECC-MTN']

# Resource types
resources = {
    'pv': {'module': 'utility', 'domain': 'd02', 'variable': 'cf'},
    'onshore': {'module': 'onshore', 'domain': 'd02', 'variable': 'cf'},
    'offshore': {'module': 'offshore', 'domain': 'd03', 'variable': 'cf'}
}

# Data paths
data_dir = Path("../../data/drought_masks")
shapefile_path = Path("../../data/load_zone_shapefiles/allLoadZones.shp")

# Month to season mapping
month_to_season = {
    1: 'JFM', 2: 'JFM', 3: 'JFM',
    4: 'AMJ', 5: 'AMJ', 6: 'AMJ',
    7: 'JAS', 8: 'JAS', 9: 'JAS',
    10: 'OND', 11: 'OND', 12: 'OND'
}

# Display simulation names for plots
sim_display_names = {
    "mpi-esm1-2-hr": "MPI-ESM1-2-HR",
    "miroc6": "MIROC6",
    "taiesm1": "TaiESM1",
    "ec-earth3": "EC-Earth3"
}

## Load Binary Masks

Load pre-computed binary masks for:
1. **High demand days** - from demand analysis (SEI-based thresholds)
2. **Resource drought days** - from renewable drought analysis (10th percentile thresholds)

All masks are binary (0/1) where 1 indicates the event is occurring.

In [ ]:
def load_demand_masks(simulation, gwl, data_dir):
    """
    Load high demand binary masks for a specific simulation and GWL.
    
    Parameters
    ----------
    simulation : str
        Climate model name
    gwl : float
        Global warming level
    data_dir : Path
        Directory containing zarr files
        
    Returns
    -------
    xr.Dataset
        Binary demand masks with dimensions (time, region)
    """
    filename = f"demand_byregion_{simulation}_gwl{gwl}_demand_mask_only.zarr"
    filepath = data_dir / filename
    
    if not filepath.exists():
        raise FileNotFoundError(f"Demand mask not found: {filepath}")
    
    ds = xr.open_zarr(filepath)
    return ds

In [ ]:
def load_resource_drought_masks(simulation, resource_key, resources, data_dir):
    """
    Load resource drought binary masks for a specific simulation and resource.
    
    Parameters
    ----------
    simulation : str
        Climate model name
    resource_key : str
        Resource identifier ('pv', 'onshore', 'offshore')
    resources : dict
        Dictionary with resource configuration
    data_dir : Path
        Directory containing zarr files
        
    Returns
    -------
    xr.Dataset
        Binary drought masks with dimensions (time, region)
    """
    config = resources[resource_key]
    
    # Map resource key to file naming convention
    resource_name_map = {
        'pv': 'pv',
        'onshore': 'windpower',
        'offshore': 'windpower'
    }
    
    resource_name = resource_name_map[resource_key]
    module = config['module']
    domain = config['domain']
    variable = config['variable']
    
    filename = f"{resource_name}_{module}_{domain}_{variable}_{simulation}_ts_regional_drought_mask.zarr"
    filepath = data_dir / filename
    
    if not filepath.exists():
        raise FileNotFoundError(f"Resource mask not found: {filepath}")
    
    ds = xr.open_zarr(filepath)
    return ds

In [ ]:
# Load all data into structured dictionaries
print("Loading binary masks...\n")

demand_masks = {}  # Structure: {gwl: {sim: dataset}}
resource_masks = {}  # Structure: {resource: {sim: dataset}}

# Load demand masks
print("Loading high demand masks:")
for gwl in target_gwls:
    print(f"  GWL {gwl}°C:", end=" ")
    demand_masks[gwl] = {}
    for sim in simulations:
        try:
            demand_masks[gwl][sim] = load_demand_masks(sim, gwl, data_dir)
            print(f"{sim}", end=" ")
        except FileNotFoundError as e:
            print(f"\n    ⚠ {e}")
    print("✓")

# Load resource drought masks
print("\nLoading resource drought masks:")
for resource_key in resources.keys():
    print(f"  {resource_key}:", end=" ")
    resource_masks[resource_key] = {}
    for sim in simulations:
        try:
            resource_masks[resource_key][sim] = load_resource_drought_masks(
                sim, resource_key, resources, data_dir
            )
            print(f"{sim}", end=" ")
        except FileNotFoundError as e:
            print(f"\n    ⚠ {e}")
    print("✓")

print("\n✓ All masks loaded successfully!")

## Region Name Harmonization

The demand masks and resource masks use slightly different naming conventions for regions. We need to standardize them before computing coincidence.

**Naming differences:**
- Demand masks: `'PGE'`, `'SDGE'`, `'WECC-MTN'` (hyphens)
- Resource masks: `'PG&E'`, `'SDG&E'`, `'WECC_MTN'` (underscores)

**Solution:** Rename resource mask regions to match demand mask conventions (using the simpler names without special characters).

In [ ]:
# Define mapping from resource mask names to demand mask names
resource_to_demand_names = {
    'PG&E': 'PGE',
    'SDG&E': 'SDGE',
    'WECC_MTN': 'WECC-MTN',
    'WECC_NW': 'WECC-NW',
    'WECC_SW': 'WECC-SW',
    'IID': 'IID',
    'LDWP': 'LDWP',
    'NCNC': 'NCNC',
    'SCE': 'SCE'
}


# Apply renaming to all resource masks
for resource_key in resource_masks.keys():
    print(f"  {resource_key}:", end=" ")
    
    for sim in simulations:
        ds = resource_masks[resource_key][sim]
        
        # Get current region values
        current_regions = ds.region.values
        
        # Build rename mapping for regions that exist in this dataset
        regions_to_rename = {str(r): resource_to_demand_names[str(r)] 
                            for r in current_regions 
                            if str(r) in resource_to_demand_names}
        
        # Apply renaming if there are regions to rename
        if regions_to_rename:
            # Create new region coordinate with mapped names
            new_regions = [resource_to_demand_names.get(str(r), str(r)) for r in current_regions]
            resource_masks[resource_key][sim] = ds.assign_coords(region=new_regions)
        
        print(f"{sim}", end=" ")
    


## Time Alignment and Data Organization

Resource masks contain full timeseries (historical + future), while demand masks are already sliced to GWL periods. We need to slice resource masks to match the GWL periods for each model.

In [ ]:
# Organize data by GWL, model, region, and resource
print("Aligning time periods and organizing data...\n")

aligned_data = {}  # Structure: {gwl: {sim: {region: {resource: mask}}}}

for gwl in target_gwls:
    aligned_data[gwl] = {}
    print(f"GWL {gwl}°C:")
    
    for sim in simulations:
        aligned_data[gwl][sim] = {}
        print(f"  {sim}:", end=" ")
        
        # Get GWL crossing period for this model
        WRF_sim_name = sim_name_dict[sim]
        model = WRF_sim_name.split("_")[1]
        ensemble_member = WRF_sim_name.split("_")[2]
        start_year, end_year = get_gwl_crossing_period(model, ensemble_member, gwl)
        
        # Get demand mask for this GWL (already time-sliced)
        demand_ds = demand_masks[gwl][sim]
        
        # Process each region
        for region in utility_regions:
            if region not in aligned_data[gwl][sim]:
                aligned_data[gwl][sim][region] = {}
            
            # Store demand mask
            if region in demand_ds.region.values:
                demand_var = list(demand_ds.data_vars)[0]
                aligned_data[gwl][sim][region]['demand'] = demand_ds[demand_var].sel(region=region)
            else:
                print(f"\n    ⚠ Region {region} not found in demand data for {sim}")
                continue
            
            # Slice and store each resource mask
            for resource_key in resources.keys():
                resource_ds = resource_masks[resource_key][sim]
                
                # Slice to GWL period
                resource_gwl = resource_ds['drought_mask'].sel(
                    time=slice(f"{start_year}", f"{end_year}")
                )
                
                # Store for this region
                if region in resource_gwl.region.values:
                    aligned_data[gwl][sim][region][resource_key] = resource_gwl.sel(region=region)
        
        print("✓")

print("\n✓ All data aligned and organized!")
print(f"  Structure: {len(target_gwls)} GWLs × {len(simulations)} models × {len(utility_regions)} regions × 4 data types (demand + 3 resources)")

## Calculate Seasonal Coincidence Metrics

For each combination of demand and resource, calculate:
- **Observed coincidence**: Days per season when both events occur
- **Marginal frequencies**: Individual event frequencies
- **Expected coincidence**: What we'd expect if events were independent
- **Statistical significance**: Chi-square test for deviation from independence

In [ ]:
def calculate_seasonal_coincidence(demand_mask, resource_mask, seasons, month_to_season):
    """
    Calculate seasonal coincidence metrics between demand and resource drought.
    
    Parameters
    ----------
    demand_mask : xr.DataArray
        Binary high demand mask (0/1)
    resource_mask : xr.DataArray
        Binary resource drought mask (0/1)
    seasons : list
        List of season names
    month_to_season : dict
        Mapping from month number to season name
        
    Returns
    -------
    dict
        Dictionary with seasonal metrics including:
        - observed coincidence days per season
        - expected coincidence (under independence)
        - marginal frequencies
        - chi-square statistic and p-value
    """
    # Ensure masks are binary integers and aligned on time dimension
    demand_mask = demand_mask.astype(int)
    resource_mask = resource_mask.astype(int)
    
    # Align both masks on the time dimension to ensure they have identical coordinates
    demand_mask, resource_mask = xr.align(demand_mask, resource_mask, join='inner')
    
    # Create season labels for each timestep using the aligned time coordinate
    months = demand_mask.time.dt.month.values
    season_labels = [month_to_season[m] for m in months]
    
    # Add season as a coordinate to both masks
    demand_mask = demand_mask.assign_coords(season=('time', season_labels))
    resource_mask = resource_mask.assign_coords(season=('time', season_labels))
    
    # Group by season
    demand_seasonal = demand_mask.groupby('season')
    resource_seasonal = resource_mask.groupby('season')
    
    results = {}
    
    for season in seasons:
        # Get masks for this season
        demand_season = demand_seasonal[season]
        resource_season = resource_seasonal[season]
        
        # Calculate observed frequencies
        both = int(((demand_season == 1) & (resource_season == 1)).sum().compute())
        demand_only = int(((demand_season == 1) & (resource_season == 0)).sum().compute())
        resource_only = int(((demand_season == 0) & (resource_season == 1)).sum().compute())
        neither = int(((demand_season == 0) & (resource_season == 0)).sum().compute())
        
        total_days = len(demand_season)
        
        # Marginal frequencies
        demand_total = both + demand_only
        resource_total = both + resource_only
        
        # Expected coincidence under independence
        # P(both) = P(demand) * P(resource)
        expected_both = (demand_total / total_days) * (resource_total / total_days) * total_days
        
        # Convert to days per season (accounting for number of years)
        # Approximate: 91.3 days per season on average
        n_years = total_days / (365.25 / 4)
        
        # Chi-square test for independence
        # 2x2 contingency table: [[both, demand_only], [resource_only, neither]]
        contingency = np.array([[both, demand_only], 
                               [resource_only, neither]])
        
        # Only compute chi-square if we have valid data
        if demand_total > 0 and resource_total > 0:
            chi2, p_value, dof, expected_freq = stats.chi2_contingency(contingency)
        else:
            chi2, p_value = np.nan, np.nan
        
        results[season] = {
            # Observed counts
            'coincident': both,
            'demand_only': demand_only,
            'resource_only': resource_only,
            'neither': neither,
            
            # Totals
            'total_days': total_days,
            'n_years': n_years,
            'demand_total': demand_total,
            'resource_total': resource_total,
            
            # Per-season averages
            'coincident_per_season': both / n_years,
            'demand_only_per_season': demand_only / n_years,
            'resource_only_per_season': resource_only / n_years,
            'demand_total_per_season': demand_total / n_years,
            'resource_total_per_season': resource_total / n_years,
            
            # Expected under independence
            'expected_coincident': expected_both,
            'expected_coincident_per_season': expected_both / n_years,
            
            # Excess coincidence
            'excess_coincident': both - expected_both,
            'excess_coincident_per_season': (both - expected_both) / n_years,
            
            # Statistical test
            'chi2': chi2,
            'p_value': p_value,
            'significant': p_value < 0.05 if not np.isnan(p_value) else False
        }
    
    return results

In [ ]:
# Calculate coincidence metrics for all combinations
print("Calculating seasonal coincidence metrics...\n")

coincidence_metrics = {}  # Structure: {gwl: {sim: {region: {resource: seasonal_metrics}}}}

for gwl in target_gwls:
    coincidence_metrics[gwl] = {}
    print(f"GWL {gwl}°C:")
    
    for sim in simulations:
        coincidence_metrics[gwl][sim] = {}
        print(f"  {sim}:", end=" ")
        
        for region in utility_regions:
            if region not in aligned_data[gwl][sim]:
                continue
                
            coincidence_metrics[gwl][sim][region] = {}
            
            # Get demand mask for this region
            demand_mask = aligned_data[gwl][sim][region]['demand']
            
            # Calculate coincidence with each resource
            for resource_key in resources.keys():
                if resource_key not in aligned_data[gwl][sim][region]:
                    continue
                    
                resource_mask = aligned_data[gwl][sim][region][resource_key]
                
                # Calculate seasonal coincidence
                metrics = calculate_seasonal_coincidence(
                    demand_mask, resource_mask, seasons, month_to_season
                )
                
                coincidence_metrics[gwl][sim][region][resource_key] = metrics
        
        print("✓")

print("\n✓ All coincidence metrics calculated!")

## Convert to DataFrame for Analysis

Flatten the nested dictionary structure into a pandas DataFrame for easier analysis and plotting.

In [ ]:
# Convert to DataFrame
rows = []

for gwl in target_gwls:
    for sim in simulations:
        for region in utility_regions:
            if region not in coincidence_metrics[gwl][sim]:
                continue
                
            for resource in resources.keys():
                if resource not in coincidence_metrics[gwl][sim][region]:
                    continue
                    
                for season in seasons:
                    metrics = coincidence_metrics[gwl][sim][region][resource][season]
                    
                    row = {
                        'gwl': gwl,
                        'simulation': sim,
                        'region': region,
                        'resource': resource,
                        'season': season,
                        **metrics  # Unpack all metrics
                    }
                    rows.append(row)

df = pd.DataFrame(rows)

print(f"✓ DataFrame created with {len(df)} rows")
print(f"  Shape: {df.shape}")
print(f"\nFirst few rows:")
df.head(10)

## Data Exploration

Quick exploration of the coincidence metrics to understand patterns and distributions.

In [ ]:
# Summary statistics
print("=== SUMMARY STATISTICS ===\n")

print("Average coincident days per season by resource:")
print(df.groupby('resource')['coincident_per_season'].mean().round(2))

print("\n\nAverage coincident days per season by GWL:")
print(df.groupby('gwl')['coincident_per_season'].mean().round(2))

print("\n\nAverage coincident days per season by season:")
print(df.groupby('season')['coincident_per_season'].mean().round(2))

print("\n\nStatistically significant cases (p < 0.05):")
sig_count = df[df['significant'] == True].shape[0]
total_count = df.shape[0]
print(f"{sig_count} / {total_count} ({100*sig_count/total_count:.1f}%)")

print("\n\nAverage excess coincidence (observed - expected) per season:")
print(df.groupby('resource')['excess_coincident_per_season'].mean().round(3))

In [ ]:
# Check for cases where observed significantly exceeds expected
print("=== EXCESS COINCIDENCE ANALYSIS ===\n")

# Filter to significant cases
df_sig = df[df['significant'] == True].copy()

print(f"Significant cases with POSITIVE excess (more than expected):")
positive_excess = df_sig[df_sig['excess_coincident_per_season'] > 0]
print(f"  Count: {len(positive_excess)} / {len(df_sig)} significant cases")
print(f"  Mean excess: {positive_excess['excess_coincident_per_season'].mean():.2f} days/season")

print(f"\nSignificant cases with NEGATIVE excess (less than expected):")
negative_excess = df_sig[df_sig['excess_coincident_per_season'] < 0]
print(f"  Count: {len(negative_excess)} / {len(df_sig)} significant cases")
print(f"  Mean excess: {negative_excess['excess_coincident_per_season'].mean():.2f} days/season")

# By resource
print("\n\nExcess coincidence by resource (significant cases only):")
for resource in ['pv', 'onshore', 'offshore']:
    resource_sig = df_sig[df_sig['resource'] == resource]
    mean_excess = resource_sig['excess_coincident_per_season'].mean()
    print(f"  {resource}: {mean_excess:.2f} days/season (n={len(resource_sig)})")

## Visualize Coincidence Patterns

### 1. Observed vs Expected Coincidence by Resource

In [ ]:
# Scatter plot: observed vs expected coincidence
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Observed vs Expected Coincident Days per Season', fontsize=16, y=1.02)

resource_names = {'pv': 'Solar PV', 'onshore': 'Onshore Wind'}

for idx, resource in enumerate(['pv', 'onshore']):
    ax = axes[idx]
    
    resource_data = df[df['resource'] == resource]
    
    # Plot by GWL
    for gwl in [0.8, 2.0]:
        gwl_data = resource_data[resource_data['gwl'] == gwl]
        
        ax.scatter(
            gwl_data['coincident_per_season'],
            gwl_data['expected_coincident_per_season'],
            alpha=0.6,
            s=50,
            label=f'GWL {gwl}°C',
            color=gwl_colors.get(gwl, 'gray')
        )
    
    # Add 1:1 line
    max_val = max(
        resource_data['expected_coincident_per_season'].max(),
        resource_data['coincident_per_season'].max()
    )
    ax.plot([0, max_val], [0, max_val], 'k--', alpha=0.3, label='1:1 (independence)')
    
    ax.set_xlabel('Observed Coincident Days/Season', fontsize=11)
    ax.set_ylabel('Expected Coincident Days/Season\n(if independent)', fontsize=11)
    ax.set_title(resource_names[resource], fontsize=13, fontweight='bold')
    ax.legend(frameon=True, loc='upper left')
    ax.grid(True, alpha=0.2)
    
    # Add note about above/below line
    ax.text(0.95, 0.05, 'Above line = less than expected\nBelow line = more than expected',
            transform=ax.transAxes, fontsize=8, ha='right', va='bottom',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.tight_layout()
plt.show()

## Regional Stacked Bar Chart: Demand Overlap with Resource Droughts

Calculate overlapping categories for each region:
- High demand only (no resource drought)
- High demand + low solar only
- High demand + low wind only  
- High demand + low both resources

In [ ]:
# Calculate stacked components for each region
print("Calculating stacked components for regional comparison...\n")

regional_stacks = []

for gwl in target_gwls:
    print(f"GWL {gwl}°C:")
    
    for sim in simulations:
        print(f"  {sim}...", end=" ")
        
        for region in utility_regions:
            if region not in aligned_data[gwl][sim]:
                continue
            
            # Check if all required data exists
            if 'demand' not in aligned_data[gwl][sim][region]:
                continue
            if 'pv' not in aligned_data[gwl][sim][region]:
                continue
            if 'onshore' not in aligned_data[gwl][sim][region]:
                continue
            
            # Get masks
            demand_mask = aligned_data[gwl][sim][region]['demand'].astype(int)
            pv_mask = aligned_data[gwl][sim][region]['pv'].astype(int)
            wind_mask = aligned_data[gwl][sim][region]['onshore'].astype(int)
            
            # Align all masks
            demand_mask, pv_mask, wind_mask = xr.align(demand_mask, pv_mask, wind_mask, join='inner')
            
            # Calculate total days per year
            total_days = len(demand_mask)
            n_years = total_days / 365.25
            
            # Calculate overlapping categories
            demand_only = int(((demand_mask == 1) & (pv_mask == 0) & (wind_mask == 0)).sum().compute())
            demand_pv_only = int(((demand_mask == 1) & (pv_mask == 1) & (wind_mask == 0)).sum().compute())
            demand_wind_only = int(((demand_mask == 1) & (pv_mask == 0) & (wind_mask == 1)).sum().compute())
            demand_both = int(((demand_mask == 1) & (pv_mask == 1) & (wind_mask == 1)).sum().compute())
            
            # Convert to days per year
            regional_stacks.append({
                'gwl': gwl,
                'simulation': sim,
                'region': region,
                'demand_only': demand_only / n_years,
                'demand_pv_only': demand_pv_only / n_years,
                'demand_wind_only': demand_wind_only / n_years,
                'demand_both': demand_both / n_years,
                'total_demand_days': (demand_only + demand_pv_only + demand_wind_only + demand_both) / n_years
            })
        
        print("✓")

df_stacks = pd.DataFrame(regional_stacks)

# Average across models for each region and GWL
df_stacks_mean = df_stacks.groupby(['region', 'gwl']).agg({
    'demand_only': 'mean',
    'demand_pv_only': 'mean',
    'demand_wind_only': 'mean',
    'demand_both': 'mean',
    'total_demand_days': 'mean'
}).reset_index()

print(f"\n✓ Stacked components calculated!")
print(f"  DataFrame shape: {df_stacks_mean.shape}")

In [ ]:
# Create stacked bar chart
fig, ax = plt.subplots(figsize=(14, 6))

# Prepare data
regions_sorted = sorted(utility_regions)
x = np.arange(len(regions_sorted))
width = 0.35

# Colors for stack categories
colors = {
    'demand_only': '#C9A4C6',
    'demand_pv_only': '#fdd835',  # Yellow - solar drought
    'demand_wind_only': '#42a5f5',  # Blue - wind drought
    'demand_both': '#e53935'  # Red - both droughts
}

# Plot for each GWL
for gwl_idx, gwl in enumerate(target_gwls):
    gwl_data = df_stacks_mean[df_stacks_mean['gwl'] == gwl]
    
    # Get values for each region
    demand_only_vals = [gwl_data[gwl_data['region'] == r]['demand_only'].values[0] 
                       if len(gwl_data[gwl_data['region'] == r]) > 0 else 0
                       for r in regions_sorted]
    demand_pv_vals = [gwl_data[gwl_data['region'] == r]['demand_pv_only'].values[0] 
                     if len(gwl_data[gwl_data['region'] == r]) > 0 else 0
                     for r in regions_sorted]
    demand_wind_vals = [gwl_data[gwl_data['region'] == r]['demand_wind_only'].values[0] 
                       if len(gwl_data[gwl_data['region'] == r]) > 0 else 0
                       for r in regions_sorted]
    demand_both_vals = [gwl_data[gwl_data['region'] == r]['demand_both'].values[0] 
                       if len(gwl_data[gwl_data['region'] == r]) > 0 else 0
                       for r in regions_sorted]
    
    offset = width * (gwl_idx - 0.5)
    
    # Stack the bars
    bottom = np.zeros(len(regions_sorted))
    
    # Only show labels for first GWL to avoid duplicate legend entries
    label_suffix = f' (GWL {gwl}°C)' if gwl_idx == 0 else ''
    
    ax.bar(x + offset, demand_only_vals, width, bottom=bottom,
           label=f'High demand only{label_suffix}' if gwl_idx == 0 else '',
           color=colors['demand_only'], alpha=0.8, edgecolor='white', linewidth=0.5)
    bottom += demand_only_vals
    
    ax.bar(x + offset, demand_pv_vals, width, bottom=bottom,
           label=f'+ Low solar only{label_suffix}' if gwl_idx == 0 else '',
           color=colors['demand_pv_only'], alpha=0.8, edgecolor='white', linewidth=0.5)
    bottom += demand_pv_vals
    
    ax.bar(x + offset, demand_wind_vals, width, bottom=bottom,
           label=f'+ Low wind only{label_suffix}' if gwl_idx == 0 else '',
           color=colors['demand_wind_only'], alpha=0.8, edgecolor='white', linewidth=0.5)
    bottom += demand_wind_vals
    
    ax.bar(x + offset, demand_both_vals, width, bottom=bottom,
           label=f'+ Low both{label_suffix}' if gwl_idx == 0 else '',
           color=colors['demand_both'], alpha=0.8, edgecolor='white', linewidth=0.5)

# Add GWL labels in the middle of bars
for gwl_idx, gwl in enumerate(target_gwls):
    gwl_data = df_stacks_mean[df_stacks_mean['gwl'] == gwl]
    offset = width * (gwl_idx - 0.5)
    
    # Calculate middle height for each region
    for region_idx, region in enumerate(regions_sorted):
        if len(gwl_data[gwl_data['region'] == region]) > 0:
            region_data = gwl_data[gwl_data['region'] == region].iloc[0]

            
            ax.text(region_idx + offset, 8, f'GWL\n{gwl}°C', 
                    ha='center', va='center', fontsize=8, fontweight='bold',
                    rotation=0, color='black',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor=gwl_colors.get(gwl, 'gray'), alpha=0.9, edgecolor='none'))

ax.set_ylabel('Days per Year', fontsize=12)
ax.set_xlabel('Region', fontsize=12)
ax.set_title('High Demand Days with Resource Drought Overlap by Region', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(regions_sorted, rotation=45, ha='right')
ax.legend(loc='upper left', frameon=True, fontsize=9)
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Create stacked bar chart
fig, ax = plt.subplots(figsize=(14, 6))

# Prepare data
regions_sorted = sorted(utility_regions)
x = np.arange(len(regions_sorted))
width = 0.35

# Colors for stack categories
colors = {
    'demand_only': '#C9A4C6',
    'demand_pv_only': '#fdd835',  # Yellow - solar drought
    'demand_wind_only': '#42a5f5',  # Blue - wind drought
    'demand_both': '#e53935'  # Red - both droughts
}

# Plot for each GWL
for gwl_idx, gwl in enumerate(target_gwls):
    gwl_data = df_stacks_mean[df_stacks_mean['gwl'] == gwl]
    
    # Get values for each region
    demand_only_vals = [gwl_data[gwl_data['region'] == r]['demand_only'].values[0] 
                       if len(gwl_data[gwl_data['region'] == r]) > 0 else 0
                       for r in regions_sorted]
    demand_pv_vals = [gwl_data[gwl_data['region'] == r]['demand_pv_only'].values[0] 
                     if len(gwl_data[gwl_data['region'] == r]) > 0 else 0
                     for r in regions_sorted]
    demand_wind_vals = [gwl_data[gwl_data['region'] == r]['demand_wind_only'].values[0] 
                       if len(gwl_data[gwl_data['region'] == r]) > 0 else 0
                       for r in regions_sorted]
    demand_both_vals = [gwl_data[gwl_data['region'] == r]['demand_both'].values[0] 
                       if len(gwl_data[gwl_data['region'] == r]) > 0 else 0
                       for r in regions_sorted]
    
    offset = width * (gwl_idx - 0.5)
    
    # Stack the bars
    bottom = np.zeros(len(regions_sorted))
    
    # Only show labels for first GWL to avoid duplicate legend entries
    label_suffix = f' (GWL {gwl}°C)' if gwl_idx == 0 else ''
    
    # ax.bar(x + offset, demand_only_vals, width, bottom=bottom,
    #        label=f'High demand only{label_suffix}' if gwl_idx == 0 else '',
    #        color=colors['demand_only'], alpha=0.8, edgecolor='white', linewidth=0.5)
    # bottom += demand_only_vals
    
    ax.bar(x + offset, demand_pv_vals, width, bottom=bottom,
           label=f'+ Low solar only{label_suffix}' if gwl_idx == 0 else '',
           color=colors['demand_pv_only'], alpha=0.8, edgecolor='white', linewidth=0.5)
    bottom += demand_pv_vals
    
    ax.bar(x + offset, demand_wind_vals, width, bottom=bottom,
           label=f'+ Low wind only{label_suffix}' if gwl_idx == 0 else '',
           color=colors['demand_wind_only'], alpha=0.8, edgecolor='white', linewidth=0.5)
    bottom += demand_wind_vals
    
    ax.bar(x + offset, demand_both_vals, width, bottom=bottom,
           label=f'+ Low both{label_suffix}' if gwl_idx == 0 else '',
           color=colors['demand_both'], alpha=0.8, edgecolor='white', linewidth=0.5)

# Add GWL labels in the middle of bars
for gwl_idx, gwl in enumerate(target_gwls):
    gwl_data = df_stacks_mean[df_stacks_mean['gwl'] == gwl]
    offset = width * (gwl_idx - 0.5)
    
    # Calculate middle height for each region
    for region_idx, region in enumerate(regions_sorted):
        if len(gwl_data[gwl_data['region'] == region]) > 0:
            region_data = gwl_data[gwl_data['region'] == region].iloc[0]

            
            ax.text(region_idx + offset, 8, f'GWL\n{gwl}°C', 
                    ha='center', va='center', fontsize=8, fontweight='bold',
                    rotation=0, color='black',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor=gwl_colors.get(gwl, 'gray'), alpha=0.9, edgecolor='none'))

ax.set_ylabel('Days per Year', fontsize=12)
ax.set_xlabel('Region', fontsize=12)
ax.set_title('High Demand Days with Resource Drought Overlap by Region', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(regions_sorted, rotation=45, ha='right')
ax.legend(loc='upper left', frameon=True, fontsize=9)
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.show()

## Spatial Maps: Change in Coincident High Demand + Low Generation Days

Create choropleth maps showing the **increase in coincident days per year** (GWL 2.0°C - 0.8°C) for each resource and season. These maps show where the problem of simultaneous high demand and low renewable generation is worsening most in the future.

In [ ]:
# Load California utility region shapefile
gdf = gpd.read_file(shapefile_path)

# Standardize region names to match our data
region_name_mapping = {
    'PG&E': 'PGE',
    'SDG&E': 'SDGE',
    'WECC_MTN': 'WECC-MTN',
    'WECC_NW': 'WECC-NW',
    'WECC_SW': 'WECC-SW',
    'IID': 'IID',
    'LDWP': 'LDWP',
    'NCNC': 'NCNC',
    'SCE': 'SCE'
}

# Apply name standardization if needed
if 'name' in gdf.columns:
    gdf['region'] = gdf['name'].replace(region_name_mapping)
elif 'region' in gdf.columns:
    gdf['region'] = gdf['region'].replace(region_name_mapping)
elif 'NAME' in gdf.columns:
    gdf['region'] = gdf['NAME'].replace(region_name_mapping)
else:
    # Try to find the right column
    print("Shapefile columns:", gdf.columns.tolist())

print(f"✓ Loaded shapefile with {len(gdf)} regions")
print(f"  Regions: {sorted(gdf['region'].unique())}")

In [ ]:
# Calculate change in coincident days by region, season, and resource
# Using the df DataFrame that has seasonal coincidence metrics

# Separate reference and future periods
df_ref = df[df['gwl'] == 0.8].copy()
df_fut = df[df['gwl'] == 2.0].copy()

# Calculate change for each model
df_change_spatial = df_ref.merge(
    df_fut,
    on=['simulation', 'region', 'resource', 'season'],
    suffixes=('_ref', '_fut')
)

df_change_spatial['change_coincident'] = (
    df_change_spatial['coincident_per_season_fut'] - 
    df_change_spatial['coincident_per_season_ref']
)

# Calculate multi-model mean change by region, season, and resource
df_change_mean = df_change_spatial.groupby(['region', 'season', 'resource']).agg({
    'change_coincident': ['mean', 'std']
}).reset_index()

df_change_mean.columns = ['region', 'season', 'resource', 'change_mean', 'change_std']

print(f"✓ Calculated spatial changes")
print(f"  Shape: {df_change_mean.shape}")
print(f"\nSample data:")
print(df_change_mean.head())

In [ ]:
# Create choropleth maps for PV
fig, axes = plt.subplots(1, 4, figsize=(20, 5), 
                         subplot_kw={'projection': ccrs.PlateCarree()})
fig.suptitle('Change in High Demand + Low Solar Days per Season (GWL 2.0°C - 0.8°C)', 
             fontsize=16, y=1.02, fontweight='bold')

# Filter to PV data
pv_change = df_change_mean[df_change_mean['resource'] == 'pv']

# Find common color scale across all seasons
vmin = pv_change['change_mean'].min()
vmax = pv_change['change_mean'].max()
# Make symmetric around zero
vmax_abs = max(abs(vmin), abs(vmax))
vmin, vmax = -vmax_abs, vmax_abs

for season_idx, season in enumerate(seasons):
    ax = axes[season_idx]
    
    # Get data for this season
    season_data = pv_change[pv_change['season'] == season]
    
    # Merge with shapefile
    gdf_plot = gdf.merge(season_data[['region', 'change_mean']], 
                         on='region', how='left')
    
    # Plot
    gdf_plot.plot(column='change_mean', ax=ax, 
                  cmap='RdBu_r', vmin=vmin, vmax=vmax,
                  edgecolor='black', linewidth=0.5,
                  legend=False, missing_kwds={'color': 'lightgray'})
    
    # Add region labels with numerical values
    for idx_region, row in gdf_plot.iterrows():

        if idx_region <5:
            if pd.notna(row['change_mean']):
                centroid = row.geometry.centroid
                ax.text(
                    centroid.x, centroid.y, 
                    f"{row['change_mean']:.1f}",
                    ha='center', va='center', 
                    fontsize=9, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7)
                )
    
    # Add state boundaries for context
    ax.coastlines(resolution='10m', linewidth=0.5, alpha=0.5)
    ax.add_feature(cfeature.STATES.with_scale('10m'), 
                   linewidth=0.5, edgecolor='gray', facecolor='none', alpha=0.3)
    
    # Set extent to California region
    ax.set_extent([-125, -114, 32, 42], crs=ccrs.PlateCarree())
    
    ax.set_title(season, fontsize=13, fontweight='bold')
    ax.set_aspect('auto')

# Add colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
sm = plt.cm.ScalarMappable(cmap='RdBu_r', 
                           norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Change in Coincident Days per Season', fontsize=11, fontweight='bold')

plt.tight_layout(rect=[0, 0, 0.91, 1])
plt.savefig('figures/high_demand_low_solar_change_choropleth.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Create choropleth maps for Onshore Wind
fig, axes = plt.subplots(1, 4, figsize=(20, 5), 
                         subplot_kw={'projection': ccrs.PlateCarree()})
fig.suptitle('Change in High Demand + Low Wind Days per Season (GWL 2.0°C - 0.8°C)', 
             fontsize=16, y=1.02, fontweight='bold')

# Filter to onshore wind data
wind_change = df_change_mean[df_change_mean['resource'] == 'onshore']

# Find common color scale across all seasons
vmin = wind_change['change_mean'].min()
vmax = wind_change['change_mean'].max()
# Make symmetric around zero
vmax_abs = max(abs(vmin), abs(vmax))
vmin, vmax = -vmax_abs, vmax_abs

for season_idx, season in enumerate(seasons):
    ax = axes[season_idx]
    
    # Get data for this season
    season_data = wind_change[wind_change['season'] == season]
    
    # Merge with shapefile
    gdf_plot = gdf.merge(season_data[['region', 'change_mean']], 
                         on='region', how='left')
    
    # Plot
    gdf_plot.plot(column='change_mean', ax=ax, 
                  cmap='RdBu_r', vmin=vmin, vmax=vmax,
                  edgecolor='black', linewidth=0.5,
                  legend=False, missing_kwds={'color': 'lightgray'})
    
    # Add region labels with numerical values
    for idx_region, row in gdf_plot.iterrows():
        if idx_region < 5:
            if pd.notna(row['change_mean']):
                centroid = row.geometry.centroid
                ax.text(
                    centroid.x, centroid.y, 
                    f"{row['change_mean']:.1f}",
                    ha='center', va='center', 
                    fontsize=9, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7)
                )
    
    # Add state boundaries for context
    ax.coastlines(resolution='10m', linewidth=0.5, alpha=0.5)
    ax.add_feature(cfeature.STATES.with_scale('10m'), 
                   linewidth=0.5, edgecolor='gray', facecolor='none', alpha=0.3)
    
    # Set extent to California region
    ax.set_extent([-125, -114, 32, 42], crs=ccrs.PlateCarree())
    
    ax.set_title(season, fontsize=13, fontweight='bold')
    ax.set_aspect('auto')

# Add colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
sm = plt.cm.ScalarMappable(cmap='RdBu_r', 
                           norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Change in Coincident Days per Season', fontsize=11, fontweight='bold')

plt.tight_layout(rect=[0, 0, 0.91, 1])
plt.savefig('figures/high_demand_low_wind_change_choropleth.png', dpi=300, bbox_inches='tight')
plt.show()


### Triple Coincidence: High Demand + Low Wind + Low Solar

Calculate change in days with ALL three conditions occurring simultaneously.

In [ ]:
# Calculate triple coincidence (high demand + low wind + low PV) by season
print("Calculating seasonal triple coincidence...\n")

triple_coincidence_data = []

for gwl in target_gwls:
    print(f"GWL {gwl}°C:")
    
    for sim in simulations:
        print(f"  {sim}...", end=" ")
        
        for region in utility_regions:
            if region not in aligned_data[gwl][sim]:
                continue
            
            # Check if all required data exists
            if 'demand' not in aligned_data[gwl][sim][region]:
                continue
            if 'pv' not in aligned_data[gwl][sim][region]:
                continue
            if 'onshore' not in aligned_data[gwl][sim][region]:
                continue
            
            # Get masks
            demand_mask = aligned_data[gwl][sim][region]['demand']
            pv_mask = aligned_data[gwl][sim][region]['pv']
            wind_mask = aligned_data[gwl][sim][region]['onshore']
            
            # Align all masks
            demand_mask, pv_mask, wind_mask = xr.align(demand_mask, pv_mask, wind_mask, join='inner')
            
            # Create triple coincidence mask
            triple_mask = (demand_mask == 1) & (pv_mask == 1) & (wind_mask == 1)
            
            # Add season coordinate
            months = triple_mask.time.dt.month.values
            season_labels = [month_to_season[m] for m in months]
            triple_mask = triple_mask.assign_coords(
                season=('time', season_labels)
            )
            
            # Calculate coincident days per season
            for season in seasons:
                season_mask = triple_mask.where(triple_mask.season == season, drop=True)
                n_years = len(season_mask) / (365.25 / 4)  # Approximate years for this season
                
                if n_years > 0:
                    coincident_days = int(season_mask.sum().compute())
                    coincident_per_season = coincident_days / n_years
                    
                    triple_coincidence_data.append({
                        'gwl': gwl,
                        'simulation': sim,
                        'region': region,
                        'season': season,
                        'triple_coincident_days': coincident_days,
                        'triple_coincident_per_season': coincident_per_season
                    })
        
        print("✓")

df_triple = pd.DataFrame(triple_coincidence_data)

print(f"\n✓ Triple coincidence calculated!")
print(f"  DataFrame shape: {df_triple.shape}")
print(f"\nSample data:")
print(df_triple.head())

### Extract Specific Dates of Triple Coincidence Events

Identify the actual dates when high demand, low wind, and low solar all occur together.

In [ ]:
# Extract actual dates when all three conditions occur
print("Extracting dates of triple coincidence events...\n")

triple_event_dates = []

for gwl in target_gwls:
    print(f"GWL {gwl}°C:")
    
    for sim in simulations:
        print(f"  {sim}...", end=" ")
        
        for region in utility_regions:
            if region not in aligned_data[gwl][sim]:
                continue
            
            # Check if all required data exists
            if 'demand' not in aligned_data[gwl][sim][region]:
                continue
            if 'pv' not in aligned_data[gwl][sim][region]:
                continue
            if 'onshore' not in aligned_data[gwl][sim][region]:
                continue
            
            # Get masks
            demand_mask = aligned_data[gwl][sim][region]['demand']
            pv_mask = aligned_data[gwl][sim][region]['pv']
            wind_mask = aligned_data[gwl][sim][region]['onshore']
            
            # Align all masks
            demand_mask, pv_mask, wind_mask = xr.align(demand_mask, pv_mask, wind_mask, join='inner')
            
            # Create triple coincidence mask
            triple_mask = (demand_mask == 1) & (pv_mask == 1) & (wind_mask == 1)
            
            # Extract dates where all three conditions are true
            event_dates = triple_mask.time[triple_mask.values].values
            
            # Add season and other info for each date
            for date in event_dates:
                # Handle cftime objects (noleap calendar from WRF)
                # Extract year, month, day directly from the cftime object
                year = date.year
                month = date.month
                day = date.day
                season = month_to_season[month]
                
                # Create a standard datetime for the date column
                date_str = f"{year:04d}-{month:02d}-{day:02d}"
                
                triple_event_dates.append({
                    'date': date_str,
                    'year': year,
                    'month': month,
                    'day': day,
                    'season': season,
                    'gwl': gwl,
                    'simulation': sim,
                    'region': region
                })
        
        print("✓")

df_triple_dates = pd.DataFrame(triple_event_dates)

print(f"\n✓ Triple coincidence dates extracted!")
print(f"  Total events: {len(df_triple_dates)}")
print(f"  Date range: {df_triple_dates['date'].min()} to {df_triple_dates['date'].max()}")
print(f"\nFirst few events:")
print(df_triple_dates.head(10))

In [ ]:
# Analyze patterns in triple coincidence dates
print("=== TRIPLE COINCIDENCE DATE PATTERNS ===\n")

# By season
print("Events by season:")
season_counts = df_triple_dates.groupby('season').size()
print(season_counts.sort_values(ascending=False))

print("\n\nEvents by month:")
month_counts = df_triple_dates.groupby('month').size()
print(month_counts.sort_values(ascending=False))

print("\n\nEvents by region (top 5):")
region_counts = df_triple_dates.groupby('region').size().sort_values(ascending=False)
print(region_counts.head())

print("\n\nEvents by GWL:")
gwl_counts = df_triple_dates.groupby('gwl').size()
print(gwl_counts)

print("\n\nChange in event frequency (GWL 2.0 vs 0.8):")
gwl_08_count = len(df_triple_dates[df_triple_dates['gwl'] == 0.8])
gwl_20_count = len(df_triple_dates[df_triple_dates['gwl'] == 2.0])
change = gwl_20_count - gwl_08_count
pct_change = (change / gwl_08_count * 100) if gwl_08_count > 0 else 0
print(f"  GWL 0.8°C: {gwl_08_count} events")
print(f"  GWL 2.0°C: {gwl_20_count} events")
print(f"  Change: +{change} events ({pct_change:.1f}% increase)")

# Example dates for one region
print("\n\nExample: Triple coincidence dates in PGE region")
print("(showing first 20 events at GWL 2.0°C)")
pge_dates = df_triple_dates[
    (df_triple_dates['region'] == 'PGE') & 
    (df_triple_dates['gwl'] == 2.0)
].sort_values('date').head(20)
if len(pge_dates) > 0:
    print(pge_dates[['date', 'season', 'simulation']].to_string(index=False))
else:
    print("  No events found")

In [ ]:
# Optional: Filter to specific region/GWL/season for detailed investigation
# Uncomment and modify as needed

# Example: Get all triple coincidence dates for SCE region in summer (JAS) at GWL 2.0
# filtered_dates = df_triple_dates[
#     (df_triple_dates['region'] == 'SCE') & 
#     (df_triple_dates['season'] == 'JAS') &
#     (df_triple_dates['gwl'] == 2.0)
# ].sort_values('date')
# print(f"Found {len(filtered_dates)} events")
# print(filtered_dates[['date', 'simulation']])

# Export to CSV if needed
# df_triple_dates.to_csv('../../data/triple_coincidence_event_dates.csv', index=False)
# print("✓ Exported to CSV")

In [ ]:
# Calculate change in triple coincidence by region and season
# Separate reference and future periods
df_triple_ref = df_triple[df_triple['gwl'] == 0.8].copy()
df_triple_fut = df_triple[df_triple['gwl'] == 2.0].copy()

# Calculate change for each model
df_triple_change = df_triple_ref.merge(
    df_triple_fut,
    on=['simulation', 'region', 'season'],
    suffixes=('_ref', '_fut')
)

df_triple_change['change_triple_coincident'] = (
    df_triple_change['triple_coincident_per_season_fut'] - 
    df_triple_change['triple_coincident_per_season_ref']
)

# Calculate multi-model mean change by region and season
df_triple_change_mean = df_triple_change.groupby(['region', 'season']).agg({
    'change_triple_coincident': ['mean', 'std']
}).reset_index()

df_triple_change_mean.columns = ['region', 'season', 'change_mean', 'change_std']

print(f"✓ Calculated triple coincidence changes")
print(f"  Shape: {df_triple_change_mean.shape}")
print(f"\nSample data:")
print(df_triple_change_mean.head(10))

In [ ]:
# Create choropleth maps for Triple Coincidence (High Demand + Low Wind + Low Solar)
fig, axes = plt.subplots(1, 4, figsize=(20, 5), 
                         subplot_kw={'projection': ccrs.PlateCarree()})
fig.suptitle('Change in High Demand + Low Wind + Low Solar Days per Season (GWL 2.0°C - 0.8°C)', 
             fontsize=16, y=1.02, fontweight='bold')

# Find common color scale across all seasons
vmin = df_triple_change_mean['change_mean'].min()
vmax = df_triple_change_mean['change_mean'].max()
# Make symmetric around zero
vmax_abs = max(abs(vmin), abs(vmax))
vmin, vmax = -vmax_abs, vmax_abs

for season_idx, season in enumerate(seasons):
    ax = axes[season_idx]
    
    # Get data for this season
    season_data = df_triple_change_mean[df_triple_change_mean['season'] == season]
    
    # Merge with shapefile
    gdf_plot = gdf.merge(season_data[['region', 'change_mean']], 
                         on='region', how='left')
    
    # Plot
    gdf_plot.plot(column='change_mean', ax=ax, 
                  cmap='RdBu_r', vmin=vmin, vmax=vmax,
                  edgecolor='black', linewidth=0.5,
                  legend=False, missing_kwds={'color': 'lightgray'})
    
    # Add region labels with numerical values
    for idx_region, row in gdf_plot.iterrows():
        if idx_region < 10:  # Only label first 5 regions to avoid clutter
            if pd.notna(row['change_mean']):
                centroid = row.geometry.centroid
                ax.text(
                    centroid.x, centroid.y, 
                    f"{row['change_mean']:.1f}",
                    ha='center', va='center', 
                    fontsize=9, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7)
                )
    
    # Add state boundaries for context
    ax.coastlines(resolution='10m', linewidth=0.5, alpha=0.5)
    ax.add_feature(cfeature.STATES.with_scale('10m'), 
                   linewidth=0.5, edgecolor='gray', facecolor='none', alpha=0.3)
    
    # Set extent to California region
    #ax.set_extent([-125, -114, 32, 42], crs=ccrs.PlateCarree())
    
    ax.set_title(season, fontsize=13, fontweight='bold')
    ax.set_aspect('auto')

# Add colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
sm = plt.cm.ScalarMappable(cmap='RdBu_r', 
                           norm=plt.Normalize(vmin=vmin, vmax=vmax))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Change in Coincident Days per Season', fontsize=11, fontweight='bold')

plt.tight_layout(rect=[0, 0, 0.91, 1])
plt.savefig('figures/high_demand_low_wind_low_solar_change_choropleth.png', dpi=300, bbox_inches='tight')
plt.show()